# Team 56: gán nháp nhãn cho ảnh đường Việt Nam (Colab / Kaggle, cần GPU)

Model lớn (COCO) + YOLOE (open-vocabulary, cho Sign / Other vehicle) đề xuất polygon cho 11 lớp, **người sửa lại trong CVAT**.
Model ở đây chỉ dùng để **gán nháp**, không phải model nộp bài (model nộp bài vẫn là YOLO26n-seg train bằng notebook của BTC).

1. Bật GPU. Kaggle: bật thêm Internet.
2. Sửa ô **Tham số**: thư mục ảnh và thư mục đầu ra.
3. Chạy lần lượt. Kết quả: `<OUT>/images`, `<OUT>/labels`, `<OUT>/autolabel_report.csv` (ảnh cần xem trước nằm ở đầu).

In [ ]:
#@title Tham số
REPO_URL = "https://github.com/MHuy21/hackathon-l2-team56.git"   # repo private: dùng https://<token>@github.com/...
IMAGES_DIR = "/content/drive/MyDrive/team56/vn_raw/batch_01"     # ảnh chưa gán (Kaggle: /kaggle/input/<dataset>/...)
OUT_DIR = "/content/drive/MyDrive/team56/drafts/batch_01"        # Kaggle: /kaggle/working/drafts/batch_01
USE_SAM = True        # tinh chỉnh biên bằng SAM 2 (chậm hơn ~2 lần, biên đẹp hơn)
LIMIT = 0             # 0 = tất cả ảnh; đặt 10 để chạy thử
SAMPLE = 0            # >0: lấy ngẫu nhiên N ảnh theo seed 61 (bộ Kaggle 49k ảnh: ví dụ 1500)

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and (IMAGES_DIR.startswith("/content/drive") or OUT_DIR.startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")
WORK = "/content" if IN_COLAB else "/kaggle/working"
REPO = f"{WORK}/team56"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.214", "pycocotools", "shapely"], check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ (rất chậm)")

In [ ]:
cmd = [sys.executable, f"{REPO}/scripts/label/autolabel.py", "--images", IMAGES_DIR, "--out", OUT_DIR]
if USE_SAM:
    cmd += ["--sam", "sam2.1_b.pt"]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if SAMPLE:
    cmd += ["--sample", str(SAMPLE)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Xem nhanh vài ảnh có độ ưu tiên duyệt cao nhất
import csv, numpy as np
from PIL import Image, ImageDraw
from IPython.display import display
sys.path.insert(0, f"{REPO}/src")
from robotaxi.classes import CLASSES
from robotaxi.yoloseg_io import read_labels
rows = list(csv.DictReader(open(f"{OUT_DIR}/autolabel_report.csv", encoding="utf-8")))
for r in rows[:4]:
    im = Image.open(f"{OUT_DIR}/images/{r['file']}").convert("RGB"); w, h = im.size
    d = ImageDraw.Draw(im)
    for inst in read_labels(f"{OUT_DIR}/labels/{os.path.splitext(r['file'])[0]}.txt"):
        pts = [tuple(p) for p in inst.poly * [w, h]]
        d.polygon(pts, outline=(255, 0, 0), width=2); d.text(pts[0], CLASSES[inst.cls], fill=(255, 255, 0))
    im.thumbnail((960, 960)); print(r["file"], "priority", r["priority"]); display(im)